# Chowdeck — Data Cleaning

Làm sạch dữ liệu gốc Chowdeck (Nigeria). Chạy file này ĐẦU TIÊN, xuất ra data/clean/chowdeck_clean.csv cho 2 notebook sau dùng.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

## ĐƯỜNG DẪN

In [2]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent  # khi chạy dạng .py
except NameError:
    BASE_DIR = Path.cwd().resolve().parent  # khi chạy trong Jupyter Notebook (đứng trong notebooks/)
RAW_PATH = BASE_DIR / 'data' / 'raw' / 'Chowdeck_Order_Delivery_Details.xlsx'
CLEAN_DIR = BASE_DIR / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## 1. ĐỌC DỮ LIỆU GỐC

In [3]:
df = pd.read_excel(RAW_PATH, sheet_name='Sheet1')
print("Shape ban đầu:", df.shape)

Shape ban đầu: (5000, 27)


## 2. CHUẨN HÓA TÊN CỘT

In [4]:
df.columns = [c.strip() for c in df.columns]  # bỏ khoảng trắng thừa (vd 'Total ' -> 'Total')

## 3. KIỂM TRA TÍNH LOGIC THỜI GIAN

In [5]:
time_cols = ['Order Received', 'Preparing Order', 'Rider Accepted', 'Order Ready',
             'Rider At Vendor', 'Rider Picked Up', 'Order Arrived', 'Order Delivered']

is_monotonic = pd.Series(True, index=df.index)
for i in range(len(time_cols) - 1):
    is_monotonic &= (df[time_cols[i]] <= df[time_cols[i + 1]])
df['time_sequence_valid'] = is_monotonic
print(f"\nSố đơn có thứ tự thời gian KHÔNG hợp lệ: {(~is_monotonic).sum()} / {len(df)}")


Số đơn có thứ tự thời gian KHÔNG hợp lệ: 1463 / 5000


## 4. KIỂM TRA OUTLIER (chỉ xem, chưa loại bỏ)

In [6]:
print("\nThống kê mô tả các cột số:")
print(df[['Unit Price', 'Distance (km)', 'Sub Total', 'Total']].describe())


Thống kê mô tả các cột số:
         Unit Price  Distance (km)      Sub Total          Total
count   5000.000000    5000.000000    5000.000000    5000.000000
mean    6762.200000       8.518440    8886.800000   10280.927600
std     7334.624087       4.143014   10605.790037   10776.467286
min      500.000000       0.500000     500.000000    1160.000000
25%     2000.000000       5.000000    3000.000000    3856.000000
50%     5000.000000       8.200000    5000.000000    6428.500000
75%     7000.000000      12.100000   10000.000000   12012.750000
max    35000.000000      16.000000  105000.000000  106675.000000


## 5. KIỂM TRA RATING HỢP LỆ

In [7]:
invalid_rating = df[(df['Rating'] < 1) | (df['Rating'] > 5)]
print(f"\nSố Rating ngoài khoảng [1,5]: {len(invalid_rating)}")


Số Rating ngoài khoảng [1,5]: 0


## 6. TẠO CỘT PHÁI SINH: THỜI GIAN XỬ LÝ ĐƠN

In [8]:
df['prep_time_min'] = (df['Order Ready'] - df['Preparing Order']).dt.total_seconds() / 60
df['delivery_time_min'] = (df['Order Delivered'] - df['Order Received']).dt.total_seconds() / 60
df['delay_min'] = (df['Order Delivered'] - df['Expected Delivery Time']).dt.total_seconds() / 60
# delay_min > 0 : giao trễ so với dự kiến | < 0 : giao sớm hơn dự kiến

## 7. TẠO CỘT PHÁI SINH: THỜI ĐIỂM ĐẶT HÀNG

In [9]:
df['order_hour'] = df['Order Received'].dt.hour
df['order_dayofweek'] = df['Order Received'].dt.day_name()
df['order_month'] = df['Order Received'].dt.to_period('M').astype(str)
df['order_year'] = df['Order Received'].dt.year

## 8. XÁC ĐỊNH CỘT REVENUE CHUẨN

In [10]:
df['revenue'] = df['Sub Total']  # Revenue của nhà hàng, KHÔNG gồm Delivery Fee/Service Fee

## 9. LỌC PHẠM VI: CHỈ GIỮ NHÓM LIÊN QUAN ẨM THỰC

In [11]:
restaurant_categories = ['Food', 'Drinks & Beverages', 'Pastries']
n_before = len(df)
df = df[df['Order Category'].isin(restaurant_categories)].copy()
n_after = len(df)
print(f"\nLọc Order Category: giữ {n_after}/{n_before} dòng "
      f"(loại bỏ Groceries & Medications, mất {n_before - n_after} dòng)")


Lọc Order Category: giữ 3061/5000 dòng (loại bỏ Groceries & Medications, mất 1939 dòng)


## 10. LOẠI CỘT KHÔNG PHỤC VỤ PHÂN TÍCH

In [12]:
df_clean = df.drop(columns=['Delivery PIN', 'Url', 'Wallet Balance'])

## 11. LƯU FILE ĐÃ LÀM SẠCH

In [13]:
df_clean.to_csv(CLEAN_DIR / 'chowdeck_clean.csv', index=False)
print(f"\nĐã lưu chowdeck_clean.csv | Shape cuối: {df_clean.shape}")
print("=== HOÀN TẤT CLEANING CHOWDECK ===")


Đã lưu chowdeck_clean.csv | Shape cuối: (3061, 33)
=== HOÀN TẤT CLEANING CHOWDECK ===
